In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 4 - WEEK 12 BAYESIAN OPTIMISATION
# Run from inside week12/
# ============================================================
#
# Function 4 has a stable high-performing local basin.
#
# Week 10 and Week 11 both gave strong results close to the
# incumbent, and recent GP residuals were within ~1 sigma.
#
# Week 12 strategy:
# - refit including Week 11
# - centre on the ACTUAL incumbent
# - tighten the local trust region
# - compare EI, posterior mean and UCB
# - inspect one-coordinate profiles
# - do not expand unless there is unusually strong evidence
# ============================================================


# ------------------------------------------------------------
# 1. Load cumulative Week 12 data
# ------------------------------------------------------------

X = np.load(
    "function4/initial_inputs.npy"
)

Y = np.load(
    "function4/initial_outputs.npy"
).reshape(-1)

best_idx = np.argmax(Y)

best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(
    best_x,
    "->",
    best_y
)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 11 calibration check
# ------------------------------------------------------------

week11_pred_mean = (
    0.4535070043533622
)

week11_pred_std = (
    0.25854242580580267
)

week11_actual = (
    0.6140414365565303
)

week11_error = (
    week11_actual
    - week11_pred_mean
)

week11_z_error = (
    week11_error
    / week11_pred_std
)

print("\n================================")
print("WEEK 11 CALIBRATION CHECK")
print("================================")

print(
    "Predicted mean:",
    week11_pred_mean
)

print(
    "Predicted std :",
    week11_pred_std
)

print(
    "Actual        :",
    week11_actual
)

print(
    "\nPrediction error:",
    week11_error
)

print(
    "\nError / predicted std:",
    week11_z_error
)


# ------------------------------------------------------------
# 3. Fit updated ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(
            1e-3,
            1e3
        )
    )
    *
    Matern(
        length_scale=np.ones(4) * 0.2,
        length_scale_bounds=(
            0.01,
            2.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(
            1e-8,
            1e-1
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(
    X,
    Y
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(
    gp.kernel_
)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0
    / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print(
    "\nARD lengthscales:"
)
print(
    lengthscales
)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(
    relative_sensitivity
)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu
        - best_y
        - xi
    )

    valid = (
        sigma > 1e-12
    )

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask]
    - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

# Strong basin, but no improvement in the last two rounds,
# so keep the next move tight.

empirical_cap = min(
    1.5 * nearest_distance,
    0.03
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print(
    "Nearest observed point:",
    nearest_distance
)

print(
    "\nEmpirical cap:",
    empirical_cap
)


# ------------------------------------------------------------
# 6. ARD-informed trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.16 * lengthscales,
    0.01,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x
    - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x
    + trust_half_width
)

print("\n================================")
print("WEEK 12 TRUST REGION")
print("================================")

print("Centre:")
print(
    best_x
)

print(
    "\nHalf-widths:"
)
print(
    trust_half_width
)

print(
    "\nLower:"
)
print(
    lower
)

print(
    "\nUpper:"
)
print(
    upper
)


# ------------------------------------------------------------
# 7. Dense local candidate pool
# ------------------------------------------------------------

rng = np.random.default_rng(
    42
)

candidates = rng.uniform(
    lower,
    upper,
    size=(
        450000,
        4
    )
)

tree = cKDTree(
    X
)

distance, _ = tree.query(
    candidates,
    k=1
)

# F4 has historically used 0.008
candidates = candidates[
    distance > 0.008
]

print(
    "\nCandidates after duplicate filtering:"
)

print(
    len(candidates)
)


# ------------------------------------------------------------
# 8. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 9. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(
    EI
)

print("\n================================")
print("PRIMARY EI")
print("================================")

print(
    "candidate =",
    candidates[ei_idx]
)

print(
    "mean =",
    mu[ei_idx]
)

print(
    "std =",
    sigma[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)


# ------------------------------------------------------------
# 10. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "mean =",
    mu[mean_idx]
)

print(
    "std =",
    sigma[mean_idx]
)


# ------------------------------------------------------------
# 11. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"beta={beta}",
        "\n candidate =",
        candidates[idx],
        "\n mean =",
        round(
            mu[idx],
            6
        ),
        "\n std =",
        round(
            sigma[idx],
            6
        ),
        "\n UCB =",
        round(
            UCB[idx],
            6
        ),
        "\n"
    )


# ------------------------------------------------------------
# 12. Distance from incumbent
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 13. Trust-region boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(
        len(x)
    ):

        if abs(
            x[j]
            - lower[j]
        ) <= tol:

            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(
            x[j]
            - upper[j]
        ) <= tol:

            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(
        status
    )


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )


# ------------------------------------------------------------
# 14. Axis-profile diagnostics
# ------------------------------------------------------------

print("\n================================")
print("AXIS PROFILE DIAGNOSTICS")
print("================================")

for j in range(4):

    values = np.linspace(
        lower[j],
        upper[j],
        30000
    )

    profile = np.tile(
        best_x,
        (
            len(values),
            1
        )
    )

    profile[:, j] = values

    profile_distance, _ = tree.query(
        profile,
        k=1
    )

    profile_valid = profile[
        profile_distance > 0.008
    ]

    if len(profile_valid) == 0:

        print(
            f"\nx{j+1}: no admissible points"
        )

        continue

    profile_mu, profile_sigma = gp.predict(
        profile_valid,
        return_std=True
    )

    idx = np.argmax(
        profile_mu
    )

    print(
        f"\nx{j+1} profile best:"
    )

    print(
        "candidate =",
        profile_valid[idx]
    )

    print(
        "mean =",
        profile_mu[idx]
    )

    print(
        "std =",
        profile_sigma[idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            profile_valid[idx]
            - best_x
        )
    )

DATA
X shape: (41, 4)
Y shape: (41,)

Current best:
[0.361985 0.41201  0.421437 0.427725] -> 0.6704983808455585

Y range:
min = -32.625660215962455
max = 0.6704983808455585
std = 9.79107451486331

WEEK 11 CALIBRATION CHECK
Predicted mean: 0.4535070043533622
Predicted std : 0.25854242580580267
Actual        : 0.6140414365565303

Prediction error: 0.1605344322031681

Error / predicted std: 0.620921041112801

GP FIT

Fitted kernel:
2.63**2 * Matern(length_scale=[1.66, 1.5, 1.39, 1.49], nu=2.5) + WhiteKernel(noise_level=0.0005)

ARD lengthscales:
[1.65603678 1.49577563 1.3901964  1.48525692]

Normalised inverse-lengthscale sensitivity:
[0.2265852  0.2508621  0.26991397 0.25263873]

EMPIRICAL LOCAL SCALE
Nearest observed point: 0.010659258088628863

Empirical cap: 0.015988887132943295

WEEK 12 TRUST REGION
Centre:
[0.361985 0.41201  0.421437 0.427725]

Half-widths:
[0.01598889 0.01598889 0.01598889 0.01598889]

Lower:
[0.34599611 0.39602111 0.40544811 0.41173611]

Upper:
[0.37797389 0.42799

In [2]:
# ============================================================
# FINAL FUNCTION 4 - WEEK 12 SELECTION
# ============================================================
#
# The local GP remains reasonably calibrated, but the previous
# two rounds made similar multi-coordinate moves without
# improving the incumbent.
#
# The Week 12 axis profiles show the strongest isolated signal
# along x2. Its predicted mean is close to the unrestricted
# highest-mean candidate relative to the model uncertainty.
#
# Therefore make a controlled one-coordinate refinement:
# decrease x2 while keeping x1, x3 and x4 at the proven
# incumbent values.
# ============================================================

week12_candidate = np.array([
    0.361985,
    0.40108336,
    0.421437,
    0.427725
])

print("================================")
print("FUNCTION 4 - WEEK 12 FINAL")
print("================================")

print("\nCandidate:")
print(week12_candidate)

# GP prediction
final_mu, final_sigma = gp.predict(
    week12_candidate.reshape(1, -1),
    return_std=True
)

print("\nPredicted mean:")
print(final_mu[0])

print("\nPredicted std:")
print(final_sigma[0])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week12_candidate - best_x
    )
)

# Check distance to ALL previous observations
nearest_dist, _ = tree.query(
    week12_candidate.reshape(1, -1),
    k=1
)

print("\nNearest observed-point distance:")
print(nearest_dist[0])

portal = "-".join(
    f"{x:.6f}"
    for x in week12_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 4 - WEEK 12 FINAL

Candidate:
[0.361985   0.40108336 0.421437   0.427725  ]

Predicted mean:
0.4810118045908087

Predicted std:
0.23983029002766657

Distance from current best:
0.010926639999999987

Nearest observed-point distance:
0.010926639999999987

Portal format:
0.361985-0.401083-0.421437-0.427725
